## ERP133459

**paper:** [no PMID](https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1) - Modulation of Avian Iridescence via Melanogenesis, 2026

**date, curator:** 2026-03-25, Sara Carsanaro

**notes**
* this is a preprint, no PMID yet
* removed all DNA seq libraries
* the data availability statement has different IDs, however those IDs do not seem to exist. also, the samples described in the methods are exactly the same as in this experiment

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,BRAIN,UBERON:0000955,brain,perfect match
1,KIDNEY,UBERON:0002113,kidney,perfect match
2,SKIN,UBERON:0000014,zone of skin,perfect match
3,LUNG,UBERON:0002048,lung,perfect match
5,TESTIS,UBERON:0000473,testis,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,juvenile,UBERON:0034919,juvenile stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "ERP133459"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [4]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 24/24 [00:24<00:00,  1.03s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [5]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,,,,,,BRAIN,juvenile,,,,M,,,9049,,,,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA1,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,,,,,,KIDNEY,juvenile,,,,M,,,9049,,,,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA5,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,,,,,,SKIN,juvenile,,,,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,,,,,,LUNG,juvenile,,,,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,,,,,,LUNG,juvenile,,,,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,,,,,,TESTIS,juvenile,,,,M,,,9049,,,,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA3,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,,,,,,SKIN,juvenile,,,,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [6]:
unique_sorted(library, "infoOrgan")

['BRAIN' 'KIDNEY' 'LUNG' 'SKIN' 'TESTIS']


In [7]:

# BRAIN
library.loc[library["infoOrgan"] == "BRAIN", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "BRAIN", "anatName"] = "brain"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "BRAIN", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "BRAIN", "anatBiologicalStatus"] = "not documented"

# KIDNEY
library.loc[library["infoOrgan"] == "KIDNEY", "anatId"] = "UBERON:0002113"
library.loc[library["infoOrgan"] == "KIDNEY", "anatName"] = "kidney"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "KIDNEY", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "KIDNEY", "anatBiologicalStatus"] = "not documented"

# LUNG
library.loc[library["infoOrgan"] == "LUNG", "anatId"] = "UBERON:0002048"
library.loc[library["infoOrgan"] == "LUNG", "anatName"] = "lung"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "LUNG", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "LUNG", "anatBiologicalStatus"] = "not documented"

# SKIN
library.loc[library["infoOrgan"] == "SKIN", "anatId"] = "UBERON:0000014"
library.loc[library["infoOrgan"] == "SKIN", "anatName"] = "zone of skin"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "SKIN", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "SKIN", "anatBiologicalStatus"] = "not documented"

# TESTIS
library.loc[library["infoOrgan"] == "TESTIS", "anatId"] = "UBERON:0000473"
library.loc[library["infoOrgan"] == "TESTIS", "anatName"] = "testis"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "TESTIS", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "TESTIS", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,,,,BRAIN,juvenile,perfect match,not documented,,M,,,9049,,,,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA1,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,,,,KIDNEY,juvenile,perfect match,not documented,,M,,,9049,,,,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA5,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,,,,SKIN,juvenile,perfect match,not documented,,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,,,,LUNG,juvenile,perfect match,not documented,,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,,,,LUNG,juvenile,perfect match,not documented,,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,UBERON:0000473,testis,,,,TESTIS,juvenile,perfect match,not documented,,M,,,9049,,,,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA3,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,,,,SKIN,juvenile,perfect match,not documented,,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [8]:
unique_sorted(library, "infoStage")

['juvenile']


In [9]:
# all
library.loc[:,'stageId'] = 'UBERON:0034919'
library.loc[:,'stageName'] = 'juvenile stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,BRAIN,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA1,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,UBERON:0034919,juvenile stage,,KIDNEY,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA5,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,TESTIS,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA3,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,,,,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'KAPA Stranded mRNA-Seq kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,BRAIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA1,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,UBERON:0034919,juvenile stage,,KIDNEY,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA5,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,TESTIS,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA3,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,,25/03/2026,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM


#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

   #libraryId       SRSId
3  ERX6997455  ERS8957806
4  ERX6997456  ERS8957806
2  ERX6997459  ERS8957808
6  ERX6997458  ERS8957808


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_16957/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-25'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,BRAIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA1,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,UBERON:0034919,juvenile stage,,KIDNEY,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA5,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA2,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,TESTIS,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA3,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,,,,SAC,2026-03-25,,,PAVCRI_RNA4,,,,juvenile,Pavo cristatus,TRANSCRIPTOMIC,RANDOM


#### comments

In [13]:
library.loc[:,'comment'] = 'still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,BRAIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
1,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,UBERON:0034919,juvenile stage,,KIDNEY,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
2,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
3,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
4,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
5,ERX6997457,ERP133459,Illumina HiSeq 4000,ERS8957807,UBERON:0000473,testis,UBERON:0034919,juvenile stage,,TESTIS,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus testis RNA,SAMEA11312953,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25
6,ERX6997458,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,"still a preprint, https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full",,,SAC,2026-03-25


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP133459,Pavo cristatus (bPavCri1) sequencing reads,"This study contains the sequencing reads used to assemble and annotate the genome of the bPavCri1 specimen, as part of the EASI-GENOMICS initiative.",SRA,,,,,,,PRJEB49014,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

7

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP133459,Pavo cristatus (bPavCri1) sequencing reads,"This study contains the sequencing reads used to assemble and annotate the genome of the bPavCri1 specimen, as part of the EASI-GENOMICS initiative.",SRA,partial,Bgee 1K,7,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB49014,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full'
experiment.loc[:,'DOI'] = '10.64898/2026.01.26.701322'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP133459,Pavo cristatus (bPavCri1) sequencing reads,"This study contains the sequencing reads used to assemble and annotate the genome of the bPavCri1 specimen, as part of the EASI-GENOMICS initiative.",SRA,partial,Bgee 1K,7,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB49014,,https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full,10.64898/2026.01.26.701322,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'removed DNAseq libraries, no PMID yet still a preprint'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP133459,Pavo cristatus (bPavCri1) sequencing reads,"This study contains the sequencing reads used to assemble and annotate the genome of the bPavCri1 specimen, as part of the EASI-GENOMICS initiative.",SRA,partial,Bgee 1K,7,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB49014,,https://www.biorxiv.org/content/10.64898/2026.01.26.701322v1.full,10.64898/2026.01.26.701322,,"removed DNAseq libraries, no PMID yet still a preprint"


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56721,SRX26064556,SRP532240,Illumina NovaSeq 6000,SRS22633731,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,After Hatch Year,perfect match,not documented,other,NA,,,40217,,,polyA,,,TRP065,SAMN43486662,,,,,,,PMID: 40421060,,,SAC,2026-03-25
56722,SRX26064551,SRP532240,Illumina NovaSeq 6000,SRS22633726,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,After Hatch Year,perfect match,not documented,other,NA,,,56313,,,polyA,,,LWC162,SAMN43486657,,,,,,,PMID: 40421060,,,SAC,2026-03-25
56723,ERX6997454,ERP133459,Illumina HiSeq 4000,ERS8957805,UBERON:0000955,brain,UBERON:0034919,juvenile stage,,BRAIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus brain RNA,SAMEA11312951,,,,,,,"still a preprint, https://www.biorxiv.org/cont...",,,SAC,2026-03-25
56724,ERX6997460,ERP133459,Illumina HiSeq 4000,ERS8957809,UBERON:0002113,kidney,UBERON:0034919,juvenile stage,,KIDNEY,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,,Pavo cristatus kidney RNA,SAMEA11312955,,,,,,,"still a preprint, https://www.biorxiv.org/cont...",,,SAC,2026-03-25
56725,ERX6997459,ERP133459,Illumina HiSeq 4000,ERS8957808,UBERON:0000014,zone of skin,UBERON:0034919,juvenile stage,,SKIN,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,2,Pavo cristatus skin RNA,SAMEA11312954,,,,,,,"still a preprint, https://www.biorxiv.org/cont...",,,SAC,2026-03-25
56726,ERX6997455,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,"still a preprint, https://www.biorxiv.org/cont...",,,SAC,2026-03-25
56727,ERX6997456,ERP133459,Illumina HiSeq 4000,ERS8957806,UBERON:0002048,lung,UBERON:0034919,juvenile stage,,LUNG,juvenile,perfect match,not documented,perfect match,M,,,9049,KAPA Stranded mRNA-Seq kit,full_length,polyA,,1,Pavo cristatus lung RNA,SAMEA11312952,,,,,,,"still a preprint, https://www.biorxiv.org/cont...",,,SAC,2026-03-25


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1118,SRP379147,Direct and indirect effects of a social supergene,To investigate the effect of supergene status ...,SRA,total,Bgee 1K,52,"KAPA Stranded RNA-Seq kit, SMARTer Ultra Low R...",full_length,GSE205680,PRJNA847170,36541826,https://onlinelibrary.wiley.com/doi/10.1111/me...,10.1111/mec.16830,,[Bgee curator notes: 2 protocols for 2 differe...
1119,SRP532240,Prevalence and Diversity of Haemosporidian-Ass...,Whole blood was collected from wild birds of t...,SRA,partial,Bgee 1K,9,,,,PRJNA1156234,40421060,https://pmc.ncbi.nlm.nih.gov/articles/PMC12105...,10.1002/ece3.71239,,removed libraries with viral infection
1120,ERP133459,Pavo cristatus (bPavCri1) sequencing reads,This study contains the sequencing reads used ...,SRA,partial,Bgee 1K,7,KAPA Stranded mRNA-Seq kit,full_length,,PRJEB49014,,https://www.biorxiv.org/content/10.64898/2026....,10.64898/2026.01.26.701322,,"removed DNAseq libraries, no PMID yet still a ..."


### add annotations to git

In [28]:
! git pull

Already up to date.


In [29]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [30]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./
	../NCBI_output/

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 8ff4747] adding annotated bulk experiment ERP133459
 2 files changed, 8 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.02 KiB | 2.02 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   637a62c..8ff4747  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push